# Test03: compare dev notes with the enriched ICD data

This notebook joins each dev note's gold ICD code to `data/final-icd-codes.csv` and displays the note alongside the matching ICD title, definition, and text. It also reports missing and duplicate codes for data-quality review.

The join uses the same uppercase/stem normalization as `score.py`. A matching code does **not** establish that the ICD description or the gold label is medically correct; review the source and clinical context separately. No model calls are made and neither input is modified.

In [6]:
import html
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from score import load_valid, norm

root = Path.cwd()
if not (root / 'score.py').is_file():
    raise FileNotFoundError('Start the notebook with the project root as its working directory.')
notes_path = root / 'data' / 'dev_notes.jsonl'  # or dev_notes_train.jsonl / dev_notes_test.jsonl
icd_path = root / 'data' / 'final-icd-codes.csv'
mms_path = root / 'data' / 'mms-2026-01.json'
focus_note_id = 'dev-002'  # change this to inspect another note

## Load notes and ICD descriptions

The ICD CSV is loaded as text to preserve code formatting. Rows without a code are counted but cannot be joined to a gold code.

In [2]:
with notes_path.open(encoding='utf-8') as file:
    notes = [json.loads(line) for line in file if line.strip()]
if not notes:
    raise ValueError(f'{notes_path} contains no notes.')
for note in notes:
    if not isinstance(note, dict) or not isinstance(note.get('id'), str) or not isinstance(note.get('text'), str) or not isinstance(note.get('gold'), list) or not note['gold'] or any(not norm(code) for code in note['gold']):
        raise ValueError('Each note must have an id, text, and non-empty gold codes.')
if len({note['id'] for note in notes}) != len(notes):
    raise ValueError('Note IDs must be unique.')

icd = pd.read_csv(icd_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
required = {'code', 'title', 'definition', 'text'}
if not required.issubset(icd.columns):
    raise ValueError(f'ICD CSV is missing columns: {sorted(required - set(icd.columns))}')
icd = icd[['code', 'title', 'definition', 'text']].rename(columns={'code': 'icd_code', 'text': 'icd_text'})
icd['normalized_code'] = icd['icd_code'].map(norm)
mms_codes = load_valid(mms_path)
print(f'Loaded {len(notes)} notes and {len(icd)} ICD CSV rows.')

Loaded 60 notes and 19004 ICD CSV rows.


## Join gold labels and check coverage

There is one row per note and gold code, or more if the CSV has multiple rows for that normalized code. Duplicate matches are retained rather than silently discarded.

In [3]:
gold_rows = pd.DataFrame([
    {'note_id': note['id'], 'note_text': note['text'], 'gold_code': code, 'normalized_code': norm(code)}
    for note in notes for code in note['gold']
])
coded_icd = icd[icd['normalized_code'].notna()].copy()
comparison = gold_rows.merge(coded_icd, how='left', on='normalized_code', indicator=True, sort=False)
comparison['csv_match'] = comparison['_merge'].eq('both')
comparison['valid_mms'] = comparison['normalized_code'].isin(mms_codes)
matched_codes = set(coded_icd['normalized_code'])
matched_gold = gold_rows['normalized_code'].isin(matched_codes)
code_counts = coded_icd['normalized_code'].value_counts()
print(f"ICD CSV rows with no code: {icd['normalized_code'].isna().sum()}")
print(f'Normalized ICD codes with multiple CSV rows: {(code_counts > 1).sum()}')
print(f"Gold labels with a CSV match: {matched_gold.sum()}/{len(gold_rows)}")
print(f"Notes with all gold codes matched: {gold_rows.assign(matched=matched_gold).groupby('note_id')['matched'].all().sum()}/{len(notes)}")
print(f"Gold labels valid in the scoring MMS: {gold_rows['normalized_code'].isin(mms_codes).sum()}/{len(gold_rows)}")
for field in ('title', 'definition', 'icd_text'):
    blank = coded_icd[field].str.strip().eq('').sum()
    print(f'Coded ICD rows with blank {field}: {blank}')

ICD CSV rows with no code: 892
Normalized ICD codes with multiple CSV rows: 0
Gold labels with a CSV match: 61/61
Notes with all gold codes matched: 60/60
Gold labels valid in the scoring MMS: 61/61
Coded ICD rows with blank title: 0
Coded ICD rows with blank definition: 0
Coded ICD rows with blank icd_text: 0


## Overview of all dev labels

Use this table to find missing matches and scan the ICD titles and description lengths. The full note and descriptions are shown for one selected note below.

In [4]:
overview = comparison[['note_id', 'gold_code', 'icd_code', 'title', 'csv_match', 'valid_mms']].copy()
overview['definition_chars'] = comparison['definition'].fillna('').str.len()
overview['icd_text_chars'] = comparison['icd_text'].fillna('').str.len()
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100):
    display(overview)
missing = gold_rows.loc[~matched_gold, ['note_id', 'gold_code']]
if not missing.empty:
    print('Gold labels with no ICD CSV match:')
    display(missing)

,note_id,gold_code,icd_code,title,csv_match,valid_mms,definition_chars,icd_text_chars
0,dev-001,1F40.Z,1F40.Z,"Malaria due to Plasmodium falciparum, unspecified",True,True,42,93
1,dev-002,CA07.0,CA07.0,"Acute upper respiratory infection, site unspecified",True,True,143,196
2,dev-002,CA00,CA00,Acute nasopharyngitis,True,True,247,270
3,dev-003,CA23.30,CA23.30,Unspecified asthma with exacerbation,True,True,197,235
4,dev-004,1A07.Z,1A07.Z,"Typhoid fever, unspecified",True,True,66,94
5,dev-005,1F23.10,1F23.10,Vulvovaginal candidosis,True,True,267,292
6,dev-006,1E90.0,1E90.0,Varicella without complication,True,True,232,264
7,dev-007,CA40.Z,CA40.Z,"Pneumonia, organism unspecified",True,True,144,177
8,dev-008,1G05,1G05,Tungiasis,True,True,433,444
9,dev-009,1B10.0,1B10.0,"Respiratory tuberculosis, confirmed",True,True,408,445


## Inspect one note in full

Change `focus_note_id` in the first code cell, then rerun that cell and this cell. All matching rows and all gold codes for that note are displayed.

In [7]:
selected = comparison[comparison['note_id'].eq(focus_note_id)]
if selected.empty:
    raise ValueError(f'Note ID not found in {notes_path}: {focus_note_id}')

def show_field(label, value):
    text = '' if pd.isna(value) else str(value)
    return f'<h4>{html.escape(label)}</h4><pre style="white-space: pre-wrap; overflow-wrap: anywhere">{html.escape(text)}</pre>'

parts = [show_field('Note ID', focus_note_id), show_field('Dev note text', selected.iloc[0]['note_text'])]
for _, row in selected.iterrows():
    parts.append(show_field('Gold ICD code', row['gold_code']))
    if not row['csv_match']:
        parts.append('<p>No matching ICD CSV row.</p>')
        continue
    for label, field in (('Matched CSV code', 'icd_code'), ('Title', 'title'), ('Definition', 'definition'), ('ICD text', 'icd_text')):
        parts.append(show_field(label, row[field]))
display(HTML(''.join(parts)))